# Baseline Model — Random Forest (Node-Level Classification)

Each node is a Bitcoin transaction. The task is binary classification: **illicit (1)** vs **licit (0)**.
Unknown transactions (class 3) are dropped. Features are the 166 node-level features.



Split,Time Steps,Purpose
Training,1 – 29,To learn the initial patterns of licit and illicit behavior.
Validation,30 – 34,To tune hyperparameters and select the best model version.
Testing,35 – 49,"To evaluate performance on ""future"" data, specifically testing the model's ability to survive the Step 43 concept drift."

In [2]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the path to your project folder
# Note: Google Drive mounts to '/content/drive/MyDrive' by default
project_path = '/content/drive/MyDrive/stat175-final-project/'

# 3. Change the current working directory
if os.path.exists(project_path):
    %cd {project_path}
    print(f"\nSuccess! Working directory set to: {os.getcwd()}")
else:
    print(f"\nError: The directory '{project_path}' was not found. Please check the spelling.")

# 4. Install dependencies from requirements.txt
if os.path.exists('requirements.txt'):
    !pip install -r requirements.txt
else:
    print("\n'requirements.txt' not found in the current directory.")

Mounted at /content/drive
/content/drive/MyDrive/stat175-final-project

Success! Working directory set to: /content/drive/MyDrive/stat175-final-project
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score, precision_recall_fscore_support
)

In [5]:
DATA_DIR = 'data'

features_df = pd.read_csv(f'{DATA_DIR}/txs_features.csv', low_memory=False)
print(f'Columns: {features_df.shape[1]}')
print(features_df.columns.tolist()[:5])  # check first few column names

classes_df = pd.read_csv(f'{DATA_DIR}/txs_classes.csv')
classes_df.columns = ['txId', 'class']

edgelist_df = pd.read_csv(f'{DATA_DIR}/txs_edgelist.csv')

df = features_df.merge(classes_df, on='txId', how='left')

print(f'Loaded {len(df):,} transactions with {df.shape[1]} columns')

Columns: 184
['txId', 'Time step', 'Local_feature_1', 'Local_feature_2', 'Local_feature_3']
Loaded 203,769 transactions with 185 columns


## 1. Preprocessing

MinMaxScale the 17 augmented (named) Bitcoin features — raw BTC values span many orders of magnitude.

In [6]:
AUGMENTED_COLS = [
    'in_txs_degree', 'out_txs_degree', 'total_BTC', 'fees', 'size',
    'num_input_addresses', 'num_output_addresses',
    'in_BTC_min', 'in_BTC_max', 'in_BTC_mean', 'in_BTC_median', 'in_BTC_total',
    'out_BTC_min', 'out_BTC_max', 'out_BTC_mean', 'out_BTC_median', 'out_BTC_total',
]
df[AUGMENTED_COLS] = MinMaxScaler().fit_transform(df[AUGMENTED_COLS])
print(f'MinMaxScaled {len(AUGMENTED_COLS)} augmented features.')

MinMaxScaled 17 augmented features.


In [7]:
# Keep only labeled transactions (class 1 = illicit, class 2 = licit)
df = df[df['class'].isin([1, 2])].copy()

# Remap: licit (class 2) -> 0, illicit (class 1) -> 1
df['label'] = df['class'].map({1: 1, 2: 0})

print(f'Labeled transactions: {len(df):,}')
print(f"  Illicit (1): {(df['label']==1).sum():,}  ({(df['label']==1).mean()*100:.1f}%)")
print(f"  Licit   (0): {(df['label']==0).sum():,}  ({(df['label']==0).mean()*100:.1f}%)")

Labeled transactions: 46,564
  Illicit (1): 4,545  (9.8%)
  Licit   (0): 42,019  (90.2%)


## 2. Temporal Train / Test Split

| Split | Time Steps |
|---|---|
| **Train** | 1 – 29 |
| **Val** | 30 – 34 |
| **Test**  | 35 – 49 |

In [8]:
# 1. Define feature columns (excluding metadata and raw IDs)
feature_cols = [c for c in df.columns if c not in ('txId', 'Time step', 'class', 'label')]

# 2. STRICT CHRONOLOGICAL SPLIT
# We split at Step 29 for Training and Step 34 for Validation
train_df = df[(df['Time step'] <= 29) & (df['label'] != -1)]  # Steps 1-29
val_df   = df[(df['Time step'] > 29) & (df['Time step'] <= 34) & (df['label'] != -1)]  # Steps 30-34
test_df  = df[(df['Time step'] > 34) & (df['label'] != -1)]  # Steps 35-49 (The Cliff)

# 3. Extract X and y
X_train, y_train = train_df[feature_cols].values, train_df['label'].values
X_val,   y_val   = val_df[feature_cols].values,   val_df['label'].values
X_test,  y_test  = test_df[feature_cols].values,  test_df['label'].values

# 4. Print stats to verify the "Black-Swan" challenge
print(f'TRAIN (1-29): {len(y_train):,} labeled nodes | illicit={y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'VAL   (30-34): {len(y_val):,} labeled nodes | illicit={y_val.sum():,} ({y_val.mean()*100:.1f}%)')
print(f'TEST  (35-49): {len(y_test):,} labeled nodes | illicit={y_test.sum():,} ({y_test.mean()*100:.1f}%)')

TRAIN (1-29): 26,381 labeled nodes | illicit=2,871 (10.9%)
VAL   (30-34): 3,513 labeled nodes | illicit=591 (16.8%)
TEST  (35-49): 16,670 labeled nodes | illicit=1,083 (6.5%)


## 3. Random Forest

In [10]:
from sklearn.model_selection import ParameterGrid

# 1. Define hyperparameter grid
rf_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 20, 30],
    'min_samples_split': [2, 5, 10],
    'class_weight': [None, 'balanced'],
}

# 2. Grid search using validation set
best_val_f1 = -1
best_params = None

for params in ParameterGrid(rf_grid):
    rf_tmp = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    rf_tmp.fit(X_train, y_train)
    y_val_pred = rf_tmp.predict(X_val)
    val_f1 = f1_score(y_val, y_val_pred, pos_label=1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_params = params

print(f"Best Val F1: {best_val_f1:.4f}")
print(f"Best Params: {best_params}")

# 3. Train final model with best params on TRAIN ONLY (steps 1-29)
rf = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# 4. Evaluate on test set (steps 35-49)
y_test_pred = rf.predict(X_test)
y_val_pred = rf.predict(X_val)

print(f"\n--- Validation Results (Steps 30-34) ---")
print(f"Val F1 Score  : {f1_score(y_val, y_val_pred, pos_label=1):.3f}")
print(f"Val Precision : {precision_score(y_val, y_val_pred, pos_label=1):.3f}")
print(f"Val Recall    : {recall_score(y_val, y_val_pred, pos_label=1):.3f}")

print(f"\n--- Final Test Results (Steps 35-49) ---")
test_f1 = f1_score(y_test, y_test_pred, pos_label=1)
print(f"Test F1 Score : {test_f1:.3f}")
print(f"Test Precision: {precision_score(y_test, y_test_pred, pos_label=1):.3f}")
print(f"Test Recall   : {recall_score(y_test, y_test_pred, pos_label=1):.3f}")

Best Val F1: 0.9707
Best Params: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 10, 'n_estimators': 200}

--- Validation Results (Steps 30-34) ---
Val F1 Score  : 0.971
Val Precision : 0.988
Val Recall    : 0.954

--- Final Test Results (Steps 35-49) ---
Test F1 Score : 0.804
Test Precision: 0.952
Test Recall   : 0.696


## 4. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Licit', 'Illicit'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Random Forest — Test Set (steps 35–49)')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True  Positives (illicit caught): {tp:,}')
print(f'False Negatives (illicit missed): {fn:,}')
print(f'False Positives (licit flagged) : {fp:,}')
print(f'True  Negatives (licit correct) : {tn:,}')

## 5. ROC and Precision-Recall Curves

In [ ]:
y_prob = rf_cv.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_roc = roc_auc_score(y_test, y_prob)
axes[0].plot(fpr, tpr, color='#e74c3c', linewidth=1.8, label=f'RF  AUC={auc_roc:.3f}')
axes[0].plot([0,1],[0,1],'k--',linewidth=0.8,label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
axes[1].plot(rec_c, prec_c, color='#e74c3c', linewidth=1.8, label=f'RF  AP={ap:.3f}')
axes[1].axhline(y_test.mean(), color='k', linestyle='--', linewidth=0.8,
                label=f'Random (AP={y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (Illicit class)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
imp = pd.Series(rf_cv.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
imp.head(20)[::-1].plot(kind='barh', ax=ax, color='steelblue', edgecolor='none')
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('Top 20 Features — Random Forest')
plt.tight_layout()
plt.show()

def feat_group(n):
    if n.startswith('Local_feature'):       return 'Local'
    elif n.startswith('Aggregate_feature'): return 'Aggregate'
    else:                                   return 'Named (BTC)'

group_imp = imp.groupby(imp.index.map(feat_group)).sum().sort_values(ascending=False)
print('\nImportance by feature group:')
print(group_imp.round(4).to_string())

## 7. Per-Time-Step Performance

Checks whether model performance is stable across the 15 test time steps.

In [ ]:
ts_test = test_df['Time step'].values
rows = []
for t in sorted(np.unique(ts_test)):
    mask = ts_test == t
    yt, yp = y_test[mask], y_test_pred[mask]
    if yt.sum() == 0:
        continue
    p, r, f, _ = precision_recall_fscore_support(yt, yp, zero_division=0)
    rows.append({'ts': int(t), 'n': int(mask.sum()), 'illicit': int(yt.sum()),
                 'precision': p[1], 'recall': r[1], 'f1': f[1]})

ts_df = pd.DataFrame(rows)
print(ts_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts_df['ts'], ts_df['f1'],        'o-',  label='F1',        color='#e74c3c')
ax.plot(ts_df['ts'], ts_df['recall'],    's--', label='Recall',    color='#3498db')
ax.plot(ts_df['ts'], ts_df['precision'], 'D:',  label='Precision', color='#2ecc71')
ax.set_xlabel('Time Step')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Per-Time-Step Performance — Test Set (steps 35–49)')
ax.legend()
plt.tight_layout()
plt.show()

# "3-Point" Graph Gain

In [ ]:
# 1. Slice only the LOCAL features (First 94 features)
# Note: In the Elliptic dataset, features 0-93 are local, 94-165 are aggregated.
X_train_local = X_combined[:, :94]
X_test_local  = X_test[:, :94]

# 2. Initialize and train the "Local-Only" Random Forest
rf_local = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_local.fit(X_train_local, y_combined)

# 3. Predict and Evaluate
y_pred_local = rf_local.predict(X_test_local)

# 4. Comparative Reporting
print(f"--- Comparison: All Features vs. Local Only ---")
f1_all = f1_score(y_test, y_test_pred, pos_label=1)
f1_local = f1_score(y_test, y_pred_local, pos_label=1)

print(f"RF (All Features)   F1: {f1_all:.4f}")
print(f"RF (Local Only)     F1: {f1_local:.4f}")
print(f"The 'Graph-Info' Gain: {f1_all - f1_local:.4f}")